In [5]:
pip install psycopg2-binary

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 30.8 MB/s  0:00:00
Note: you may need to restart the kernel to use updated packages.


In [20]:
import psycopg2
import pandas as pd

In [17]:
# POŁĄCZENIE Z BAZĄ
conn = psycopg2.connect(host='localhost', database="postgres", user="postgres", password="admin", port=5432)

In [23]:
cur = conn.cursor()

In [28]:
cur.execute("""create table IF NOT EXISTS players (
  player_id serial primary key,
  first_name text not null,
  last_name text not null,
  height numeric not null,
  weight numeric not null,
  salary numeric not null
);
""")

In [25]:
conn.commit()

In [ ]:
cur.execute(""" insert into players (first_name,last_name,height,weight, salary) values
('Marian', 'Nowak', 1.90, 80, 10000),
('Jan', 'Kowalski', 1.80, 75, 150000),
('Chuck', 'Norris', 1.70, 58, 300000);
""")
conn.commit()

In [ ]:
cur.execute("""SELECT * FROM players;""")
print(cur.fetchall())

[(1, 'Marian', 'Nowak', Decimal('1.90'), Decimal('80'), Decimal('10000')), (2, 'Jan2', 'Kowalski2', Decimal('1.80'), Decimal('75'), Decimal('150000')), (3, 'Chuck', 'Norris', Decimal('1.70'), Decimal('58'), Decimal('300000'))]


In [19]:
cur.execute("""SELECT * FROM players;""")
for row in cur.fetchall():
    print(row)

(1, 'Marian', 'Nowak', Decimal('1.90'), Decimal('80'), Decimal('11000.0'))
(2, 'Jan', 'Kowalski', Decimal('1.80'), Decimal('75'), Decimal('165000.0'))
(3, 'Chuck', 'Norris', Decimal('1.70'), Decimal('58'), Decimal('330000.0'))


In [30]:
df = pd.read_sql("SELECT * FROM players;", conn)
df


/var/folders/14/lsx5gcnn58q23ht_xjqydrnw0000gn/T/ipykernel_6245/2408499550.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql("SELECT * FROM players;", conn)


,player_id,first_name,last_name,height,weight,salary
0,1,Marian,Nowak,1.9,80.0,11000.0
1,2,Jan,Kowalski,1.8,75.0,165000.0
2,3,Chuck,Norris,1.7,58.0,330000.0


In [31]:
df2 = pd.read_sql("SELECT * FROM players2;", conn)
df2

/var/folders/14/lsx5gcnn58q23ht_xjqydrnw0000gn/T/ipykernel_6245/664251732.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df2 = pd.read_sql("SELECT * FROM players2;", conn)


,player_id,first_name,last_name,height,weight,salary
0,1,Marian,Nowak,1.9,80.0,10000.0
1,2,Jan2,Kowalski2,1.8,75.0,150000.0
2,3,Chuck,Norris,1.7,58.0,300000.0


In [36]:
import hashlib

# Dla df1
df['md5_hash'] = (df.iloc[:, 0].astype(str) + df.iloc[:, 1].astype(str)).apply(
    lambda x: hashlib.md5(x.encode()).hexdigest()
)

# Dla df2
df2['md5_hash'] = (df2.iloc[:, 0].astype(str) + df2.iloc[:, 1].astype(str)).apply(
    lambda x: hashlib.md5(x.encode()).hexdigest()
)

In [37]:
df

,player_id,first_name,last_name,height,weight,salary,md5_hash
0,1,Marian,Nowak,1.9,80.0,11000.0,dcf73420a30aa32dde8066a33828a33a
1,2,Jan,Kowalski,1.8,75.0,165000.0,7e3864761227abd208c855e2dc0b178d
2,3,Chuck,Norris,1.7,58.0,330000.0,4f10533899e3101fe530f6e12ebce56d


In [38]:
df2

,player_id,first_name,last_name,height,weight,salary,md5_hash
0,1,Marian,Nowak,1.9,80.0,10000.0,dcf73420a30aa32dde8066a33828a33a
1,2,Jan2,Kowalski2,1.8,75.0,150000.0,869b69b732328d3e18f84675ba8db728
2,3,Chuck,Norris,1.7,58.0,300000.0,4f10533899e3101fe530f6e12ebce56d


In [42]:
df['md5_hash'].sort_values() == df2['md5_hash'].sort_values()

2     True
1    False
0     True
Name: md5_hash, dtype: bool

In [29]:
# # zmiana wartości w tabeli
# cur.execute("""UPDATE players SET salary = salary + (salary * 0.1);""")
# conn.commit()

In [16]:
# NA KONIEC PRACY Z BAZĄ DANYCH
cur.close()
conn.close()